In [12]:
import os
from kaggle_secrets import UserSecretsClient


secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")

print("W&B key loaded!")

W&B key loaded!


In [13]:
import os
import numpy as np
import pandas as pd
import librosa
import wandb
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from tqdm import tqdm

print("All imports done!")

All imports done!


In [14]:
BASE = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'


wandb.init(project="23f3003600-t12026", name="model1-cnn")

print("Paths and W&B ready!")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Paths and W&B ready!


In [15]:
# Collect all audio files with their genre labels
data = []
genres_path = os.path.join(BASE, 'genres_stems')

for genre in os.listdir(genres_path):
    genre_folder = os.path.join(genres_path, genre)
    if not os.path.isdir(genre_folder):
        continue
    for song_folder in os.listdir(genre_folder):
        song_path = os.path.join(genre_folder, song_folder)
        audio_file = os.path.join(song_path, 'other.wav')
        if os.path.exists(audio_file):
            data.append((audio_file, genre))

train_df = pd.DataFrame(data, columns=['filepath', 'genre'])

print(f"Total files found: {len(train_df)}")
print(train_df['genre'].value_counts())

Total files found: 1000
genre
disco        100
metal        100
reggae       100
blues        100
rock         100
classical    100
jazz         100
hiphop       100
country      100
pop          100
Name: count, dtype: int64


In [16]:
# Convert genre names to numbers
# CNN needs numbers not strings
le = LabelEncoder()
train_df['label'] = le.fit_transform(train_df['genre'])

print("Genre to number mapping:")
for i, genre in enumerate(le.classes_):
    print(f"  {genre} → {i}")

NUM_CLASSES = len(le.classes_)
print(f"\nTotal genres: {NUM_CLASSES}")

Genre to number mapping:
  blues → 0
  classical → 1
  country → 2
  disco → 3
  hiphop → 4
  jazz → 5
  metal → 6
  pop → 7
  reggae → 8
  rock → 9

Total genres: 10


In [17]:
def get_melspec(file_path, sr=22050, duration=30, n_mels=64, n_fft=2048, hop=512):
   
    try:
        
        y, sr = librosa.load(file_path, sr=sr, duration=duration)
        
        
        target_len = sr * duration
        if len(y) < target_len:
            y = np.pad(y, (0, target_len - len(y)))
        else:
            y = y[:target_len]
        
        
        mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels,
                                              n_fft=n_fft, hop_length=hop)
        
        
        mel_db = librosa.power_to_db(mel, ref=np.max)
        
       
        mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-6)
        
        return mel_db.astype(np.float32)
    
    except:
       
        return np.zeros((n_mels, 1292), dtype=np.float32)



sample_mel = get_melspec(train_df['filepath'][0])
print(f"Mel-spectrogram shape: {sample_mel.shape}")


Mel-spectrogram shape: (64, 1292)


In [22]:
class AudioDataset(Dataset):
   
    def __init__(self, filepaths, labels):
        self.filepaths = filepaths
        self.labels = labels
    
    def __len__(self):
        
        return len(self.filepaths)
    
    def __getitem__(self, idx):
        
        mel = get_melspec(self.filepaths[idx])
        
        
        mel = torch.tensor(mel).unsqueeze(0)
        
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return mel, label

In [21]:

X_train, X_val, y_train, y_val = train_test_split(
    train_df['filepath'].values,
    train_df['label'].values,
    test_size=0.2,
    random_state=42,
    stratify=train_df['label'].values
)


train_dataset = AudioDataset(X_train, y_train)
val_dataset = AudioDataset(X_val, y_val)


train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

Train batches: 50
Val batches: 13


In [23]:
class SimpleCNN(nn.Module):
    """
    A simple CNN with 3 convolutional layers.
    
    Conv layer = finds patterns in the spectrogram image
    MaxPool = shrinks the image (keeps important info)
    Linear = final classification layer
    """
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()
        
        
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2, 2)
        
       
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2, 2)
        
        
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(2, 2)
        
        
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        
        
        self.fc = nn.Linear(64, num_classes)
    
    def forward(self, x):
        
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.pool3(self.relu3(self.conv3(x)))
        x = self.global_pool(x)          
        x = x.view(x.size(0), -1)        
        x = self.fc(x)                   
        return x



device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = SimpleCNN(num_classes=NUM_CLASSES).to(device)
print(model)

Using device: cuda
SimpleCNN(
  (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu1): ReLU()
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu2): ReLU()
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu3): ReLU()
  (pool3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (global_pool): AdaptiveAvgPool2d(output_size=1)
  (fc): Linear(in_features=64, out_features=10, bias=True)
)


In [24]:
#loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

NUM_EPOCHS = 10

print("Starting training...")

for epoch in range(NUM_EPOCHS):
    
    
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0
    
    for mels, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}"):
        mels = mels.to(device)
        labels = labels.to(device)
        
        
        outputs = model(mels)
        
        
        loss = criterion(outputs, labels)
        
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        preds = outputs.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)
    
    train_acc = train_correct / train_total
    
    
    model.eval()
    val_preds_all = []
    val_labels_all = []
    
    with torch.no_grad():
        for mels, labels in val_loader:
            mels = mels.to(device)
            outputs = model(mels)
            preds = outputs.argmax(dim=1).cpu().numpy()
            val_preds_all.extend(preds)
            val_labels_all.extend(labels.numpy())
    
    val_f1 = f1_score(val_labels_all, val_preds_all, average='macro')
    
    
    wandb.log({
        "epoch": epoch+1,
        "train_loss": train_loss / len(train_loader),
        "train_acc": train_acc,
        "val_f1": val_f1
    })
    
    print(f"Epoch {epoch+1}: Loss={train_loss/len(train_loader):.4f} | Train Acc={train_acc:.4f} | Val F1={val_f1:.4f}")

print("Training complete!")

Starting training...


Epoch 1/10: 100%|██████████| 50/50 [02:40<00:00,  3.21s/it]


Epoch 1: Loss=2.2963 | Train Acc=0.1300 | Val F1=0.0741


Epoch 2/10: 100%|██████████| 50/50 [01:50<00:00,  2.20s/it]


Epoch 2: Loss=2.1598 | Train Acc=0.2013 | Val F1=0.1562


...

Epoch 8: Loss=1.9664 | Train Acc=0.2675 | Val F1=0.2472


Epoch 9/10: 100%|██████████| 50/50 [01:44<00:00,  2.10s/it]


Epoch 9: Loss=1.9263 | Train Acc=0.3050 | Val F1=0.1915


Epoch 10/10: 100%|██████████| 50/50 [01:47<00:00,  2.15s/it]


Epoch 10: Loss=1.9192 | Train Acc=0.3025 | Val F1=0.2412
Training complete!


In [28]:

test_df = pd.read_csv(os.path.join(BASE, 'test.csv'))

model.eval()
test_preds = []

with torch.no_grad():
    for _, row in test_df.iterrows():
        file_path = os.path.join(BASE, row['filename'])
        mel = get_melspec(file_path)
        mel_tensor = torch.tensor(mel).unsqueeze(0).unsqueeze(0).to(device)
        output = model(mel_tensor)
        pred = output.argmax(dim=1).item()
        test_preds.append(pred)


test_genres = le.inverse_transform(test_preds)

submission = pd.DataFrame({
    'id': test_df['id'],
    'genre': test_genres
})

submission.to_csv('submission.csv', index=False)
print("Submission saved!")
print(submission['genre'].value_counts())


torch.save(model.state_dict(), 'cnn_model.pth')
print("Model saved!")

wandb.finish()

Submission saved!
genre
disco        1028
metal         906
classical     509
pop           270
jazz          191
hiphop         59
rock           30
reggae         27
Name: count, dtype: int64
Model saved!


epoch,▁▂▃▃▄▅▆▆▇█
train_acc,▁▄▅▆▅▆▆▇██
train_loss,█▅▄▃▃▂▂▂▁▁
val_f1,▁▄▄▆▃▅▅█▆█
epoch,10
train_acc,0.3025
train_loss,1.91919
val_f1,0.24121
